## Import Library

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV, GridSearchCV, train_test_split
from sklearn.linear_model import LassoCV
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.metrics import make_scorer
from sklearn.feature_selection import SelectFromModel
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import lightgbm as lgb
import xgboost as xgb
import catboost as cb
import seaborn as sns
from scipy.stats import uniform, randint


## Model Building and Data Handling

### Parameter Grid

#### Random Search Grid

In [2]:
param_distributions1 = {
    "LightGBM": {
        "num_leaves": [20, 31, 40, 50, 60],
        "learning_rate": [0.01, 0.03, 0.05, 0.07, 0.1],
        "n_estimators": [50, 100, 150, 200],
        "min_child_samples": [5, 10, 20, 50],
        "subsample": [0.7, 0.8, 0.9, 1.0],
        "colsample_bytree": [0.7, 0.8, 0.9, 1.0],
        "reg_alpha": [0, 0.1, 0.5, 1.0],
        "reg_lambda": [0, 0.1, 0.5, 1.0]
    },
    "XGBoost": {
        "max_depth": [3, 5, 7, 10],
        "learning_rate": [0.01, 0.03, 0.05, 0.07, 0.1],
        "n_estimators": [50, 100, 150, 200],
        "min_child_weight": [1, 3, 5, 10],
        "subsample": [0.7, 0.8, 0.9, 1.0],
        "colsample_bytree": [0.7, 0.8, 0.9, 1.0],
        "gamma": [0, 0.1, 0.3, 0.5],
        "reg_alpha": [0, 0.1, 0.5, 1.0],
        "reg_lambda": [0, 0.1, 0.5, 1.0]
    },
    "CatBoost": {
        "depth": [4, 6, 8, 10],
        "learning_rate": [0.01, 0.03, 0.05, 0.07, 0.1],
        "iterations": [50, 100, 150, 200],
        "l2_leaf_reg": [1, 3, 5, 10],
        "border_count": [32, 64, 128],
        "bagging_temperature": [0, 0.5, 1, 2],
        "random_strength": [1, 5, 10],
        "colsample_bylevel": [0.7, 0.8, 0.9, 1.0]
    }
}

#### Grid Search Grid

### Define scoring using myscore

In [3]:
def weighted_mae(y_true, y_pred, weights):
    return np.sum(weights * np.abs(y_true - y_pred)) / np.sum(weights)
weighted_mae_scorer = make_scorer(weighted_mae, greater_is_better=False)

### Do PCA on data

In [13]:
def PCA_transformer(df_train, df_test):
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(df_train.drop(columns = ['sales']))
    X_test_scaled = scaler.transform(df_test)
    pca_full = PCA().fit(X_train_scaled)
    cumulative_variance = np.cumsum(pca_full.explained_variance_ratio_)
    optimal_components = np.argmax(cumulative_variance >= 0.8) + 1
    print(f"Optimal PCA Components: {optimal_components}")
    pca_optimal = PCA(n_components=optimal_components)
    X_train_pca = pca_optimal.fit_transform(X_train_scaled)
    X_test_pca = pca_optimal.transform(X_test_scaled)
    X_train_df = pd.DataFrame(X_train_pca, columns=[f'PC{i+1}' for i in range(optimal_components)])
    X_train_df['sales'] = df_train['sales'].values
    X_test_df = pd.DataFrame(X_test_pca, columns=[f'PC{i+1}' for i in range(optimal_components)])
    return X_train_df, X_test_df

### Train and Compare Models

In [5]:
def evaluate_models(models, X, weights):
    train, test = train_test_split(X, test_size=0.2, shuffle=False)  # Ensures time order is maintained
    X_train, y_train = train.drop(columns=['sales']), train['sales']
    X_test, y_test = test.drop(columns=['sales']), test['sales']
    results = {}
    for model_name, model in models.items():
        print(f"Tuning {model_name} with RandomizedSearchCV...")
        search = RandomizedSearchCV(model, param_distributions1[model_name], n_iter=50, scoring=weighted_mae_scorer , n_jobs=-1)
        search.fit(X_train, y_train)
        best_model = search.best_estimator_
        print(f"Best parameters for {model_name}: {search.best_params_}")
        best_model.fit(X_train, y_train)
        y_pred = best_model.predict(X_test)
        results[model_name] = weighted_mae(y_test, y_pred, weights)
        print(f"{model_name} MAE: {results[model_name]:.4f}")
    return pd.DataFrame.from_dict(results, orient = 'index', columns = ['MAE'])



## Model Training

### Get weight

In [6]:
# read from "test_weights.csv" using read.csv
weights = pd.read_csv("test_weights.csv")

### Import Data

In [7]:
# read from "processed_sales_test.csv" and processed_sales_train.csv using read.csv
sales_test = pd.read_csv("processed_sales_test.csv")
sales_train = pd.read_csv("processed_sales_train.csv")


In [8]:
sales_test = sales_test.drop(columns = ['date'])
sales_train = sales_train.drop(columns = ['date'])
# sales_test = sales_test.drop(columns = ['warehouse', 'holiday_name'])

In [9]:
sales_test.columns
sales_train.columns

Index(['unique_id', 'total_orders', 'sales', 'sell_price_main',
       'product_unique_id', 'shops_closed', 'winter_school_holidays',
       'school_holidays', 'year', 'month', 'day', 'day_of_week',
       'new_years_day', 'international_womens_day', 'good_friday',
       'holy_saturday', 'easter_day', 'easter_monday', 'labour_day',
       'mother's_day', 'cyrila_a_metodej', 'jan_hus', 'den_ceske_statnosti',
       'den_vzniku_samostatneho_ceskoslovenskeho_statu',
       'den_boje_za_svobodu_a_demokracii', 'christmas_eve',
       '1st_christmas_day', '2nd_christmas_day', 'den_osvobozeni',
       'memorial_day_of_the_republic',
       'memorial_day_for_the_victims_of_the_communist_dictatorships',
       'memorial_day_for_the_victims_of_the_holocaust', 'whit_sunday',
       'whit_monday', 'national_defense_day', 'day_of_national_unity',
       'independent_hungary_day', 'state_foundation_day',
       'memorial_day_for_the_martyrs_of_arad',
       'memorial_day_of_the_1956_revolution', 'a

### Train Model

In [14]:
df_train, df_test = PCA_transformer(sales_train, sales_test)
results_df = evaluate_models({"LightGBM": lgb.LGBMRegressor(), "XGBoost": xgb.XGBRegressor(objective="reg:squarederror"), "CatBoost": cb.CatBoostRegressor(verbose=0)}, df_train, weights)
print("Model Comparison:")
print(results_df)
sns.barplot(x=results_df.index, y=results_df['MAE'])
plt.title("Model Comparison (MAE)")
plt.show()

MemoryError: Unable to allocate 978. MiB for an array with shape (128245094,) and data type float64